# Asistente en N8N

En la clase de hoy van a reanudar el chatbot que iniciamos la clase pasada pero le vamos a añadir las habilidades de **modificar nuestro calendario** de Gmail y **entender notas de voz**. Su resultado final se debe ver así.

<center>
<img src="img/Asistente_n8n.png" width=90%>
</center>

> Nota: Para este tutorial vamos a suponer que usted ya tiene creada la arquitectura de chatbot que vimos la clase pasada totalmente funcional. 

> Nota2: Es probable que deba actualizar los tokens de WhatsApp de esta sección:

<center>
<img src="img/Meta10.png" width=70%>
</center>


Nuestro tutorial se va a dividir en dos grandes secciones:
1. Procesar notas de voz.
2. Crear herramientas para manipular nuestro calendario de Gmail.

## 1. Procesar notas de voz.
Nuestro workflow comienza con un trigger de WhatsApp que se activa cada vez que el número de testeo recibe un mensaje, no importa si es una nota de voz, una imagen o un mensaje de texto. 

El primer paso es crear un `Switch` que va manipular el flujo de la información que llega de WhatsApp para conducir las notas de voz a un pipeline especializado para procesar audio.
### 1.1. Añadir un `Switch`
1. Abra el `node panel` presionando su tecla `tab` o presionando el botón con un `+` en la esquina superior derecha. En la barra de búsqueda escriba "Switch" y seleccione la primera opción.
2. Asocie el `Switch` para que sea lo que reciba el `WhatsApp trigger`. 
<center>
<img src="img/switch2.png" width=20%>
</center>

3. Abra el `Switch` y configurelo de la siguiente manera:
<center>
<img src="img/switch.png" width=90%>
</center>

Si corre el nodo anterior (WhatsApp trigger) podrá constatar que dentro de la respuesta existe un elemento llamado "type" (`messages > messages[0] > type`). Este elemento puede tomar los valores de "text", "audio" o "image". En este tutorial vamos a especificar dos canales, uno para cuando el input del modelo es texto y otro para cuando es audio.

4. Note que del `switch` salen dos conexiones. `0` es la correspondiente a la primera condición `type == text` y `1` es la correspondiente a la segunda condición `type == audio`. Nos vamos a concentrar de la que sale de `1`.

5. Para probar que todo esté en orden, presione el botón `Test workflow` y envíe una nota de voz al número de prueba. Si hace todo correctamente, dentro de WhatsApp Trigger debería poder ver algo similar a esto:
<center>
<img src="img/prueba_whatsapp.png" width=70%>
</center>

### 1.2. Descargar audio.
Como puede ver en el output de WhatsApp trigger, lo que la API nos envía NO es el audio directamente sino un ID del audio. Vamos a hacer otro llamado de la API para que nos envíen un link para descargar el audio a través de un `request`. Ven lo útil que son ;)?.

1. Abra el `node panel` presionando su tecla `tab` o presionando el botón con un `+` en la esquina superior derecha. En la barra de búsqueda escriba `WhatsApp Business Cloud`, seleccione la primera opción y luego escoja la opción `Download media`.
2. Conecte la salida `1` del Switch a `Download media`.
3. Puede arrastar el elemento `messages > messages[0] > audio > id` del Input o escribir `{{ $json.messages[0].audio.id }}` en el campo `Media ID`. La configuración se debería ver así:
<center>
<img src="img/download_audio.png" width=70%>
</center>

La parte que nos interesa del output es la URL.
4. Presionamos el `+` que sale del nodo de Download media. En la barra de búsqueda escriba `HTTP Request` y seleccione la primera opción.
5. Configure la ventana de HTTP request de la siguiente manera:
<center>
<img src="img/http_request.png" width=70%>
</center>

Esta configuración tiene dos partes. La primera es obtener la URL del nodo anterior con `{{ $json.url }}`. La segunda es seleccionar una autenticación predefinida y utilizar la opción de WhatsApp.

### 1.3. Transcribir el audio usando Whisper
6. Presionamos el `+` que sale del nodo de HTTP Request. En la barra de búsqueda escriba `OpenAI`, seleccione la primera opción y luego escoja la opción `Transcribe a recording`.
7. La configuración para el modelo que transcribe audio a texto (llamado `Whisper`) es la siguiente:
<center>
<img src="img/whisper.png" width=70%>
</center>

## 2. Crear herramientas para manipular nuestro calendario de Gmail
Lo primero que vamos a hacer es añadir un if para que el AI Agent solo se active si al workflow le llega un texto o un audio (recuerden que también podrían llegar imagenes, videos o stickers).
### 2.1. Conectar el AI Agent a los inputs.
1. Abra el `node panel` presionando su tecla `tab` o presionando el botón con un `+` en la esquina superior derecha. En la barra de búsqueda escriba `If` y seleccione la primera opción. Conecte la salida del modelo de transcripción de OpenAI y el Switch al `If`. Luego conecte el `true` del `If` al AI Agent. Su flujo se debería ver así:
<center>
<img src="img/flujo_if.png" width=70%>
</center>

2. Note que el output del flujo cuando se envía un mensaje desde el `Switch` es `$json.messages[0].text.body`, mientras que el output que sale del modelo de transcripción de OpenAI cuando se envía una nota de vos es `$json.text`. Por ende, vamos a crear una condición que sea que uno de los dos outputs sea no vacío.
<center>
<img src="img/if_condicion.png" width=70%>
</center>

3. Ahora vamos a configurar el AI Agent configurando el input `Prompt (User Message)`. Dado que tenemos dos tipos de inputs, vamos a crear un `if-else` para que el modelo tome el input que no sea vacío. Para hacerlo vamos a utilizar `javascript`. En la casilla escriba `{{ $json.text ? $json.text : $json.messages[0].text.body }}` que se podría traducir como `condición ? valor_si_verdadero : valor_si_falso`. En Python se vería como:

```python
# Si el input es una nota de voz, reciba la transcripción de OpenAI
if input == "audio":
    prompt = transcripcion_openai # $json.text
# Si no, el input debe ser un mensaje de texto
else:
    prompt = mensaje_de_whatsapp # $json.messages[0].text.body
```

<center>
<img src="img/prompt_if.png" width=20%>
</center>

### 2.2. Crear herramientas.
Vamos a crear 4 herramientas que el agente va a poder acceder para interactuar con `Gmail Calendar`: leer calendario, crear eventos, eliminar eventos y modificar eventos.

1. Creemos la primera herramienta: **Create event**. Para crear cada herramienta vamos a presionar el `+` que dice `Tool` debajo del AI Agent.
<center>
<img src="img/add_tool.png" width=70%>
</center>

2. **Importante**. Le cambiamos el nombre a la herramienta por **Create event**.
<center>
<img src="img/change_name.png" width=20%>
</center>

3. Luego creamos una credencial para Google Calendar. Es MUY fácil, solo debemos presionar `Create new credential` y luego `Sign in with Google`.
<center>
<img src="img/sign_in1.png" width=20%>
<img src="img/sign_in2.png" width=30%>
</center>

4. Luego en `Tool Description` seleccione la opción `Set Manually` y escriba algo como "Usa esta herramienta cuando el usuario desee crear un nuevo evento, agendar una cita o reunión." Esto es clave porque es el texto que va a ver el LLM de OpenAI para decidir cuando usar esta herramienta.
5. En `Resource` seleccione `Event`.
6. En `Operation` seleccione `Create`.
7. En `Calendar` seleccione `From list` y escoja el calendario en el cual quiere que se creen los eventos. Generalmente se escoge el calendario que es su misma dirección de correo (i.e `su_correo@gmail.com`).
8. Para las opciones `Start` y `End` se selecciona el botón que define la hora de inicio y final automáticamente usando el LLM. Añadir una descripción sobre que es el campo también es clave:
<center>
<img src="img/start_ia.png" width=30%>
<img src="img/start_exp.png" width=20%>
</center>
9. Finalmente añada los campos adicionales que considere convenientes, por ejemplo "Description" y "Summary":
<center>
<img src="img/additional_fields.png" width=40%>
</center>


10. Cree las demás herramientas siguiendo los mismos pasos y siguiendo las imagenes:
- **Delete event**
<center>
<img src="img/delete_event.png" width=40%>
</center>

- **Get all events**
<center>
<img src="img/get_all_events.png" width=40%>
</center>

- **Modify events**
<center>
<img src="img/modify1.png" width=40%>
</center>
<center>
<img src="img/modify2.png" width=40%>
</center>

11. Finalmente construya un `System Message` para el AI Agent. Una sugerencia podría ser:

```
Eres un asistente llamado Séneca, especializado en interpretar mensajes recibidos por WhatsApp para gestionar citas en Google Calendar. Entre tus funciones están:
- Responder preguntas generales.
- Manejar la agenda del usuario creando, modificando y eliminando eventos con base en la información de su calendario y las instrucciones que se te den.

Contexto actual:
- Fecha y hora: {{ $now }}
- Zona horaria: America/Bogotá (UTC-5)
- Utiliza la información de los últimos mensajes de la conversación para tener el contexto completo de la interacción.

Tu objetivo es:
- Comprender lo que el usuario desea hacer respecto a su calendario.
- Identificar la intención correcta.
- Extraer todos los detalles relevantes del mensaje.
- Generar una respuesta conversacional, clara, breve y con tono amable.

En caso de que el usuario desee que manipules su agenda, sigue estas instrucciones:

1. Consulta los eventos actuales de la semana usando la herramienta "Get all events".

2. Luego, dependiendo de la intención del usuario:

- Si el usuario desea agendar un nuevo evento:
  - Revisa que no existan incompatibilidades en su calendario.
  - Si hay conflictos (es decir, si el evento se cruza con otro ya agendado), notifícalo al usuario, ofrécele sugerencias (por ejemplo, mover uno de los eventos o agendar de todos modos) y pregúntale qué desea hacer.
  - Si no hay conflictos, utiliza la herramienta "Create event" para añadir el evento al calendario.

- Si el usuario desea eliminar un evento:
  - Encuentra el evento correspondiente en su agenda.
  - Utiliza la herramienta "Delete event".

- Si el usuario desea modificar un evento:
  - Encuentra el evento en su agenda.
  - Utiliza la herramienta "Modify event".

Ejemplos de mensajes e intenciones:
- “Agenda una reunión con Laura mañana a las 4” → "Create event"
- “Pasemos la cita del lunes para el miércoles” → "Modify event"
- “Cancela la reunión de hoy” → "Delete event"
- “¿Qué tengo esta semana?” → "Get all events"

Tu respuesta debe:
- Ser breve, clara y con lenguaje natural, como si chatearas por WhatsApp.
- Usar emojis, pero con moderación.
- Incluir solo texto conversacional. No devuelvas código ni estructuras técnicas como JSON.
- Confirmar la acción realizada, incluyendo fecha, hora y título si están disponibles.
- Pedir amablemente información faltante (como hora, fecha o título) si el mensaje es incompleto o ambiguo.

Datos a extraer cuando el usuario desea asistencia con su agenda (si están presentes):
- Título del evento → summary
- Fecha y hora de inicio → startTime (formato ISO 8601)
- Fecha y hora de finalización → endTime (calculado si se conoce la duración)
- Duración → si no se menciona, asume 1 hora
- Descripción → description (opcional)

Interpretaciones comunes de fechas:
- “mañana a las 10am” → día siguiente a {{ $now }}, 10:00 a.m.
- “el próximo lunes” → el lunes siguiente a {{ $now }}
- “dentro de 8 días” → {{ $now }} + 8 días
- “el mes que viene” → mismo día de {{ $now }} pero en el próximo mes
- “este viernes por la tarde” → si el viernes ya pasó, asume el próximo viernes por la tarde

Ejemplos de respuestas bien formateadas:
- Evento creado: “Listo, agendé tu cita ‘Reunión con Laura’ para mañana a las 4:00 p.m. 😊”
- Evento reprogramado: “He actualizado tu cita. Ahora es el miércoles a las 3:00 p.m.”
- Evento eliminado: “He cancelado tu cita programada para hoy. Si necesitas otra, dime 👍”
- Consulta: “Tienes una cita ‘Reunión con equipo’ el viernes a las 10:00 a.m.”

Si el mensaje es ambiguo o no se entiende claramente lo que el usuario quiere hacer, responde con una pregunta amable que ayude a clarificar la intención.
```

<center>
<img src="img/system_message1.png" width=40%>
</center>